# Gap investigation and time-series segmentation

This notebook investigates sampling gaps, proposes diagnostic explanations, and creates gap-bounded segments. The classifications are hypotheses for review—not slowdown labels.

### 1. Import tools, define paths, and fingerprint inputs

**What the code does:** Defines repository-relative paths and records SHA-256 checksums for the cleaned dataset and both time-audit reports.  
**Why it is needed:** Explicit paths make the workflow reproducible, while checksums verify that all inputs remain unchanged.  
**How to interpret the output:** It shows the exact files being read and their fingerprints before analysis.

In [ ]:
from pathlib import Path
import hashlib
import math

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_metrics.csv"
TIME_AUDIT_PATH = PROJECT_ROOT / "reports" / "time_structure_audit.csv"
RUN_SUMMARY_INPUT_PATH = PROJECT_ROOT / "reports" / "run_summary.csv"
GAP_REPORT_PATH = PROJECT_ROOT / "reports" / "gap_investigation.csv"
SEGMENT_REPORT_PATH = PROJECT_ROOT / "reports" / "segment_summary.csv"
SEGMENTED_DATA_PATH = PROJECT_ROOT / "data" / "interim" / "segmented_metrics.csv"

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

input_paths = [DATASET_PATH, TIME_AUDIT_PATH, RUN_SUMMARY_INPUT_PATH]
missing_inputs = [str(path) for path in input_paths if not path.is_file()]
if missing_inputs:
    raise FileNotFoundError(f"Required inputs are missing: {missing_inputs}")

input_hashes_before = {path: sha256_file(path) for path in input_paths}
for path, fingerprint in input_hashes_before.items():
    print(f"{path}: {fingerprint}")

### 2. Load all inputs without modifying them

**What the code does:** Reads the cleaned metrics and the two prior audit reports, then validates the required columns.  
**Why it is needed:** The gap investigation depends on trusted sequence keys, timestamps, and diagnostic metrics, while the earlier reports provide comparison context.  
**How to interpret the output:** The three shapes confirm what was loaded; no source table is edited.

In [ ]:
raw_metrics = pd.read_csv(DATASET_PATH)
time_audit_input = pd.read_csv(TIME_AUDIT_PATH)
run_summary_input = pd.read_csv(RUN_SUMMARY_INPUT_PATH)

required_metric_columns = {
    "machine_id", "run_id", "timestamp", "cpu_pct", "ram_pct", "swap_pct",
    "disk_latency_ms", "context_switches_per_s", "missed_deadline",
    "sample_reliable", "sensor_errors_json"
}
missing_metric_columns = required_metric_columns - set(raw_metrics.columns)
if missing_metric_columns:
    raise KeyError(f"Required metric columns are missing: {sorted(missing_metric_columns)}")

original_shape = raw_metrics.shape
original_columns = raw_metrics.columns.tolist()
print(f"Cleaned metrics: {raw_metrics.shape}")
print(f"Prior time issues: {time_audit_input.shape}")
print(f"Prior run summary: {run_summary_input.shape}")

### 3. Parse timestamps safely and sort only a copy

**What the code does:** Creates a deep copy, parses timestamps as UTC using `errors='coerce'`, preserves source-row identity, and stably sorts by machine, run, and time.  
**Why it is needed:** Chronological gaps require ordered timestamps, but the cleaned source must remain untouched and invalid timestamps must not crash the audit.  
**How to interpret the output:** A zero invalid count means every row can be positioned in time; sorting occurs only in `sorted_metrics`.

In [ ]:
analysis_metrics = raw_metrics.copy(deep=True)
analysis_metrics["_source_row"] = np.arange(len(analysis_metrics))
analysis_metrics["_timestamp_dt"] = pd.to_datetime(
    analysis_metrics["timestamp"], errors="coerce", utc=True
)

sorted_metrics = analysis_metrics.sort_values(
    ["machine_id", "run_id", "_timestamp_dt", "_source_row"],
    kind="mergesort",
    na_position="last",
).copy()

group_keys = ["machine_id", "run_id"]
sorted_metrics["previous_timestamp"] = sorted_metrics.groupby(group_keys)["_timestamp_dt"].shift(1)
sorted_metrics["gap_seconds"] = (
    sorted_metrics["_timestamp_dt"] - sorted_metrics["previous_timestamp"]
).dt.total_seconds()

invalid_timestamp_count = int(sorted_metrics["_timestamp_dt"].isna().sum())
print(f"Rows in sorted analysis copy: {len(sorted_metrics):,}")
print(f"Invalid timestamps: {invalid_timestamp_count:,}")
print(f"Source DataFrame remains {raw_metrics.shape}")

### 4. Calculate a normal interval and threshold for every run

**What the code does:** Calculates each run's median positive time difference and sets its threshold to `max(5 × run median, 10 seconds)`.  
**Why it is needed:** Machines and runs can have different normal cadences; a per-run rule avoids treating ordinary variation as a gap, while the 10-second floor prevents an overly sensitive threshold.  
**How to interpret the output:** Any interval strictly above the displayed run threshold is treated as a segmentation gap.

In [ ]:
positive_intervals = sorted_metrics.loc[sorted_metrics["gap_seconds"].gt(0)]
run_intervals = (
    positive_intervals.groupby(group_keys)["gap_seconds"]
    .median()
    .rename("run_median_interval_seconds")
    .reset_index()
)
run_intervals["gap_threshold_seconds"] = np.maximum(
    5.0 * run_intervals["run_median_interval_seconds"], 10.0
)

sorted_metrics = sorted_metrics.merge(run_intervals, on=group_keys, how="left", validate="many_to_one")
sorted_metrics["is_detected_gap"] = sorted_metrics["gap_seconds"].gt(
    sorted_metrics["gap_threshold_seconds"]
)

display(run_intervals)
print(f"Detected gaps using per-run thresholds: {int(sorted_metrics['is_detected_gap'].sum())}")

### 5. Define transparent diagnostic classification rules

**What the code does:** Defines pressure signals and maps each gap to `small_sampling_delay`, `possible_slowdown`, `likely_pause_or_sleep`, or `unknown`.  
**Why it is needed:** Gap duration alone cannot establish slowdown. The rules use the preceding 30-second window and remain explicit hypotheses for manual review.  
**How to interpret the output:** `possible_slowdown` means strong pre-gap evidence exists; it is not a target. Several-minute gaps without pressure suggest pause/sleep, near-threshold gaps suggest sampling delay, and insufficient evidence stays unknown.

In [ ]:
WINDOW_SECONDS = 30
PRESSURE_RULES = {
    "cpu": "30s CPU mean ≥80% or maximum ≥95%",
    "ram": "30s RAM mean ≥90%",
    "swap": "30s swap mean ≥90%",
    "disk": "30s disk-latency mean ≥50 ms",
    "deadlines": "at least 2 missed deadlines or a 30s missed-deadline rate ≥25%",
}

def classify_gap(gap_seconds, threshold, window_rows, summary):
    main_means = [
        summary.get("cpu_pct_30s_mean"), summary.get("ram_pct_30s_mean"),
        summary.get("swap_pct_30s_mean"), summary.get("disk_latency_ms_30s_mean"),
        summary.get("context_switches_per_s_30s_mean"),
    ]
    evidence_insufficient = window_rows < 2 or all(pd.isna(value) for value in main_means)
    if evidence_insufficient:
        return "unknown", "Fewer than two usable pre-gap rows or all main window metrics are missing."

    pressure_signals = []
    if summary["cpu_pct_30s_mean"] >= 80 or summary["cpu_pct_30s_max"] >= 95:
        pressure_signals.append("cpu_pressure")
    if summary["ram_pct_30s_mean"] >= 90:
        pressure_signals.append("ram_pressure")
    if summary["swap_pct_30s_mean"] >= 90:
        pressure_signals.append("swap_pressure")
    if summary["disk_latency_ms_30s_mean"] >= 50:
        pressure_signals.append("disk_latency_pressure")
    repeated_deadlines = (
        summary["missed_deadline_30s_count"] >= 2
        or summary["missed_deadline_30s_rate"] >= 0.25
    )
    if repeated_deadlines:
        pressure_signals.append("repeated_missed_deadlines")

    strong_pressure = len(pressure_signals) >= 2 or repeated_deadlines
    if strong_pressure:
        return "possible_slowdown", "Strong pre-gap evidence: " + ", ".join(pressure_signals)
    if gap_seconds >= 300:
        return "likely_pause_or_sleep", "Gap is at least five minutes and the preceding window lacks strong pressure evidence."
    if gap_seconds <= max(2.0 * threshold, 30.0):
        return "small_sampling_delay", "Gap is close to the run threshold and the preceding window lacks strong pressure evidence."
    return "unknown", "Gap is longer than a small delay but lacks enough evidence for slowdown or a several-minute pause."

print("Diagnostic rules (proposals only):")
for signal, rule in PRESSURE_RULES.items():
    print(f"- {signal}: {rule}")

### 6. Investigate every detected gap and its preceding 30 seconds

**What the code does:** Captures timestamps and metrics immediately before each gap, summarizes the same main metrics over the preceding 30 seconds, and applies the diagnostic rules.  
**Why it is needed:** Sustained conditions are more informative than a single sample, and preserving both lets a reviewer inspect the evidence behind each proposal.  
**How to interpret the output:** Each row is one detected gap; window statistics describe pre-gap behavior, while `diagnostic_class` and `classification_evidence` explain the proposal.

In [ ]:
main_metrics = ["cpu_pct", "ram_pct", "swap_pct", "disk_latency_ms", "context_switches_per_s"]
shift_columns = main_metrics + ["missed_deadline", "sample_reliable", "sensor_errors_json"]
for column in shift_columns:
    sorted_metrics[f"{column}_before"] = sorted_metrics.groupby(group_keys)[column].shift(1)

gap_records = []
detected_gap_rows = sorted_metrics.loc[sorted_metrics["is_detected_gap"]]

for _, gap_row in detected_gap_rows.iterrows():
    before_time = gap_row["previous_timestamp"]
    window_start = before_time - pd.Timedelta(seconds=WINDOW_SECONDS)
    window = sorted_metrics.loc[
        sorted_metrics["machine_id"].eq(gap_row["machine_id"])
        & sorted_metrics["run_id"].eq(gap_row["run_id"])
        & sorted_metrics["_timestamp_dt"].between(window_start, before_time, inclusive="both")
    ]

    record = {
        "machine_id": gap_row["machine_id"],
        "run_id": gap_row["run_id"],
        "timestamp_before_gap": before_time.isoformat(),
        "timestamp_after_gap": gap_row["_timestamp_dt"].isoformat(),
        "gap_seconds": float(gap_row["gap_seconds"]),
        "run_median_interval_seconds": float(gap_row["run_median_interval_seconds"]),
        "gap_threshold_seconds": float(gap_row["gap_threshold_seconds"]),
        "window_30s_row_count": int(len(window)),
    }
    for column in main_metrics:
        record[f"{column}_before"] = gap_row[f"{column}_before"]
        record[f"{column}_30s_mean"] = window[column].mean()
        record[f"{column}_30s_max"] = window[column].max()

    record["missed_deadline_before"] = gap_row["missed_deadline_before"]
    record["sample_reliable_before"] = gap_row["sample_reliable_before"]
    record["sensor_errors_json_before"] = gap_row["sensor_errors_json_before"]
    record["missed_deadline_30s_count"] = int(window["missed_deadline"].fillna(0).sum())
    record["missed_deadline_30s_rate"] = window["missed_deadline"].mean()
    record["sample_reliable_30s_rate"] = window["sample_reliable"].mean()
    record["sensor_error_rows_30s_count"] = int(
        window["sensor_errors_json"].fillna("{}").astype(str).str.strip().ne("{}").sum()
    )

    diagnostic_class, evidence = classify_gap(
        record["gap_seconds"], record["gap_threshold_seconds"], len(window), record
    )
    record["diagnostic_class"] = diagnostic_class
    record["classification_evidence"] = evidence
    gap_records.append(record)

gap_investigation = pd.DataFrame(gap_records)
print(f"Investigated gaps: {len(gap_investigation)}")
display(gap_investigation.head())

### 7. Start a new segment after every detected gap

**What the code does:** Cumulatively counts gaps inside each machine/run and creates a globally descriptive `segment_id`; the first segment is `seg_001`, and the row after a gap starts the next segment.  
**Why it is needed:** Label windows must never cross missing time, run boundaries, or machine boundaries.  
**How to interpret the output:** Every original row remains present exactly once, and segment IDs change only after a detected gap within the same run.

In [ ]:
sorted_metrics["segment_number"] = (
    sorted_metrics.groupby(group_keys)["is_detected_gap"].cumsum().astype(int) + 1
)
sorted_metrics["segment_id"] = (
    sorted_metrics["machine_id"].astype(str)
    + "__" + sorted_metrics["run_id"].astype(str)
    + "__seg_" + sorted_metrics["segment_number"].astype(str).str.zfill(3)
)

assert len(sorted_metrics) == len(raw_metrics)
assert sorted_metrics["_source_row"].nunique() == len(raw_metrics)
assert not sorted_metrics.duplicated(["machine_id", "run_id", "_source_row"]).any()

print(f"Rows preserved: {len(sorted_metrics):,}")
print(f"Segments created: {sorted_metrics['segment_id'].nunique():,}")
display(sorted_metrics[["machine_id", "run_id", "timestamp", "gap_seconds", "is_detected_gap", "segment_id"]].head())

### 8. Summarize segments and assess 5- and 10-minute usability

**What the code does:** Calculates segment boundaries, duration, row count, and median interval. A segment is safe when its duration is strictly longer than the horizon and it contains at least 80% of the observations expected from its median cadence.  
**Why it is needed:** Duration alone can hide sparse data; the observation-coverage rule checks that a segment has enough measurements for a meaningful future window.  
**How to interpret the output:** `safe_for_5min_label` and `safe_for_10min_label` identify where later label definitions may be evaluated without crossing a detected gap.

In [ ]:
sorted_metrics["segment_interval_seconds"] = (
    sorted_metrics.groupby(["machine_id", "run_id", "segment_id"])["_timestamp_dt"]
    .diff()
    .dt.total_seconds()
)

segment_summary = (
    sorted_metrics.groupby(["machine_id", "run_id", "segment_id"], dropna=False)
    .agg(
        start_time=("_timestamp_dt", "min"),
        end_time=("_timestamp_dt", "max"),
        row_count=("_source_row", "size"),
        median_interval_seconds=("segment_interval_seconds", lambda values: values[values.gt(0)].median()),
        segment_number=("segment_number", "first"),
    )
    .reset_index()
)
segment_summary["duration_seconds"] = (
    segment_summary["end_time"] - segment_summary["start_time"]
).dt.total_seconds()
segment_summary["number_of_gaps_before_segment"] = segment_summary["segment_number"] - 1

def segment_is_safe(row, horizon_seconds):
    median_interval = row["median_interval_seconds"]
    if pd.isna(median_interval) or median_interval <= 0:
        return False
    expected_rows = horizon_seconds / median_interval + 1
    minimum_rows = math.ceil(0.80 * expected_rows)
    return bool(row["duration_seconds"] > horizon_seconds and row["row_count"] >= minimum_rows)

segment_summary["safe_for_5min_label"] = segment_summary.apply(lambda row: segment_is_safe(row, 300), axis=1)
segment_summary["safe_for_10min_label"] = segment_summary.apply(lambda row: segment_is_safe(row, 600), axis=1)
segment_summary = segment_summary[[
    "machine_id", "run_id", "segment_id", "start_time", "end_time", "duration_seconds",
    "row_count", "median_interval_seconds", "number_of_gaps_before_segment",
    "safe_for_5min_label", "safe_for_10min_label"
]]

display(segment_summary.head())

### 9. Save and validate the three requested outputs

**What the code does:** Saves the gap report, segment summary, and a time-sorted copy of every original column plus `segment_id`, then reads them back for validation.  
**Why it is needed:** Persisted outputs support later review and label design while assertions prevent row loss, accidental source overwrite, or missing gap records.  
**How to interpret the output:** The printed paths and row counts confirm successful creation; the segmented dataset must have exactly the source row count and one additional column.

In [ ]:
GAP_REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
SEGMENTED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

segmented_metrics = sorted_metrics[original_columns + ["segment_id"]].copy()
gap_investigation.to_csv(GAP_REPORT_PATH, index=False)
segment_summary.to_csv(SEGMENT_REPORT_PATH, index=False)
segmented_metrics.to_csv(SEGMENTED_DATA_PATH, index=False)

saved_gaps = pd.read_csv(GAP_REPORT_PATH)
saved_segments = pd.read_csv(SEGMENT_REPORT_PATH)
saved_metrics = pd.read_csv(SEGMENTED_DATA_PATH)

assert len(saved_gaps) == int(sorted_metrics["is_detected_gap"].sum())
assert len(saved_segments) == sorted_metrics["segment_id"].nunique()
assert len(saved_metrics) == len(raw_metrics)
assert saved_metrics.columns.tolist() == original_columns + ["segment_id"]
assert all(path.resolve() != DATASET_PATH.resolve() for path in [GAP_REPORT_PATH, SEGMENT_REPORT_PATH, SEGMENTED_DATA_PATH])

print(f"Created {GAP_REPORT_PATH} ({len(saved_gaps)} gaps)")
print(f"Created {SEGMENT_REPORT_PATH} ({len(saved_segments)} segments)")
print(f"Created {SEGMENTED_DATA_PATH} ({len(saved_metrics):,} rows)")

### 10. Display the final summary and verify input integrity

**What the code does:** Counts diagnostic classes and usable segments, identifies runs with zero detected gaps versus runs needing review, and rechecks all input checksums.  
**Why it is needed:** This is the decision-oriented handoff for future label-definition work and proof that the investigation remained non-destructive.  
**How to interpret the output:** The dataset is ready only for label-definition experiments inside safe segments; diagnostic classes must not be treated as final targets.

In [ ]:
gap_class_counts = gap_investigation["diagnostic_class"].value_counts().sort_index()
run_gap_counts = (
    sorted_metrics.groupby(group_keys)["is_detected_gap"].sum().astype(int).rename("detected_gap_count").reset_index()
)
continuous_runs = run_gap_counts.loc[run_gap_counts["detected_gap_count"].eq(0), "run_id"].tolist()
review_runs = run_gap_counts.loc[run_gap_counts["detected_gap_count"].gt(0), "run_id"].tolist()
safe_5min_count = int(segment_summary["safe_for_5min_label"].sum())
safe_10min_count = int(segment_summary["safe_for_10min_label"].sum())
ready_for_label_definition_work = safe_5min_count > 0 and safe_10min_count > 0

input_hashes_after = {path: sha256_file(path) for path in input_paths}
inputs_unchanged = input_hashes_before == input_hashes_after

print("FINAL GAP INVESTIGATION SUMMARY")
print(f"Detected gaps: {len(gap_investigation)}")
print("Gap classes:")
display(gap_class_counts.to_frame("count"))
print(f"Segments: {len(segment_summary)}")
print(f"Segments safe for 5-minute labels: {safe_5min_count}")
print(f"Segments safe for 10-minute labels: {safe_10min_count}")
print(f"Runs with mostly continuous data (zero detected gaps): {continuous_runs}")
print(f"Runs needing review: {review_runs}")
print(f"Ready for label-definition work inside safe segments: {ready_for_label_definition_work}")
print(f"All input files unchanged: {inputs_unchanged}")
print("Reminder: diagnostic gap classes are not slowdown labels.")

assert raw_metrics.shape == original_shape and raw_metrics.columns.tolist() == original_columns
assert inputs_unchanged, "An input file changed during the investigation."
